In [ ]:
pip install torch transformers datasets pillow scikit-learn nltk rouge-score matplotlib seaborn torchvision open_clip_torch pandas numpy tqdm opencv-python albumentations gensim peft bitsandbytes accelerate rouge-score bitsandbytes

In [ ]:
pip install flash-attn --no-build-isolation

In [ ]:
!pip install -U bitsandbytes transformers accelerate peft

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
cd '/content/drive/MyDrive/Latest_Med-VQA'

In [ ]:
# @title
import os
import json
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# BiomedCLIP with OpenCLIP
import open_clip
from open_clip import create_model_from_pretrained, get_tokenizer

from transformers import (
    AutoModel, AutoTokenizer,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

import copy
from random import shuffle, seed
import sys
import os.path
import argparse
import glob
import scipy.io
import pdb
import string
import h5py
import nltk
from nltk.tokenize import word_tokenize
import gensim
import re
import tensorflow as tf
import keras
import warnings
from keras.applications.vgg16 import VGG16
import keras.activations
import keras.backend as kbe
from keras.callbacks import EarlyStopping
import tensorflow.keras.layers as layers
from keras.layers import Activation, Add, Concatenate, Conv1D, Dense, Dropout, Embedding, Softmax
from keras.layers import Input, GlobalMaxPooling1D, Lambda, Multiply, RepeatVector, Reshape
from keras.layers import BatchNormalization
from keras.models import Model
from keras.regularizers import l2
import pickle
from pprint import pprint
from tensorflow.keras.utils import plot_model
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Dropout, Embedding, LSTM, GlobalMaxPooling1D, SpatialDropout1D,Flatten,Concatenate
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout, Concatenate, Flatten, Conv2D, Reshape, Permute
from tensorflow.keras.utils import to_categorical, plot_model
from tensorflow.keras.callbacks import ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns
from tqdm.auto import tqdm
from transformers import (
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    Trainer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
from open_clip import create_model_from_pretrained, get_tokenizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    get_linear_schedule_with_warmup
)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
warnings.filterwarnings('ignore')
nltk.download('punkt')
dropout_rate = 0.4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(42)

In [ ]:
# @title
# 2. DATASET CLASS (FIXING TOKEN MISMATCH)
# ---------------------------------------------------------
class LlavaVQADataset(Dataset):
    def __init__(self, df, processor, max_length=4096): # Set max_length to a more manageable value
        self.df = df
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(IMG_DIR, row['image_name'])
        image = Image.open(image_path).convert("RGB")

        # Generative prompt (No CoT)
        prompt = f"USER: <image>\n{row['question']}\nASSISTANT:"
        answer = str(row['answer'])

        # Processor handles image expansion. If truncated, you get the 'image token mismatch' error.
        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        # Tokenize labels
        labels = self.processor.tokenizer(
            answer,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        ).input_ids

        return {
            "input_ids": inputs.input_ids.squeeze(),
            "pixel_values": inputs.pixel_values.squeeze(),
            "image_sizes": inputs.image_sizes.squeeze(),
            "labels": labels.squeeze()
        }
def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    image_sizes = torch.stack([item['image_sizes'] for item in batch])

    # Handle variable number of image patches
    pixel_values = [item['pixel_values'] for item in batch]

    # Find the max number of patches in this batch
    max_patches = max([p.shape[0] for p in pixel_values])
    channels, height, width = pixel_values[0].shape[1:]

    # Pad pixel_values to the max number of patches
    padded_pixel_values = []
    for p in pixel_values:
        num_patches = p.shape[0]
        if num_patches < max_patches:
            # Create padding tensor of zeros
            padding = torch.zeros((max_patches - num_patches, channels, height, width), dtype=p.dtype)
            p = torch.cat([p, padding], dim=0)
        padded_pixel_values.append(p)

    return {
        "input_ids": input_ids,
        "labels": labels,
        "pixel_values": torch.stack(padded_pixel_values),
        "image_sizes": image_sizes
    }

In [ ]:
# @title
import torch
import gc
from transformers import (
    LlavaProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup
import huggingface_hub
from rouge_score import rouge_scorer # Import rouge_scorer

# Clean up previous memory
gc.collect()
torch.cuda.empty_cache()

# BakLLaVA is Mistral-based, LLaVA-arch, and usually open (no login required)
model_id = "llava-hf/bakllava-v1-hf"

# Initialize Processor (No token needed)
processor = LlavaProcessor.from_pretrained(model_id)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

# 1. DATA PREPARATION
# ---------------------------------------------------------
JSON_PATH = "./archive/VQA_RAD Dataset Public.json"
IMG_DIR = "./archive/VQA_RAD Image Folder/"

with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)

# Create Train/Val/Test splits (80/10/10)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(test_df, test_size=0.5, random_state=42)

print(f"Dataset Split: Train({len(train_df)}), Val({len(val_df)}), Test({len(test_df)})")

from torch.utils.data import WeightedRandomSampler
import numpy as np

# Force answers to strings
y_train = train_df['answer'].astype(str).values

# Calculate class counts and weights
classes, class_sample_count = np.unique(y_train, return_counts=True)
weight = 1. / class_sample_count
samples_weight = np.array([weight[np.where(classes == t)[0][0]] for t in y_train])

samples_weight = torch.from_numpy(samples_weight).double()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

# 3. MODEL & CONFIGURATION
# ---------------------------------------------------------
# Re-initialize processor for consistency
processor = LlavaProcessor.from_pretrained(model_id)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load Model (No token needed)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# PEFT Setup
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Send model to device
model.to(device)

# 4. TRAINING WITH EARLY STOPPING
# ---------------------------------------------------------
# Redefine dataset to ensure compatibility with standard LlavaProcessor
class LlavaVQADataset(Dataset):
    def __init__(self, df, processor, max_length=1024):
        self.df = df
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(IMG_DIR, row['image_name'])
        image = Image.open(image_path).convert("RGB")

        # Standard LLaVA prompt format
        prompt = f"USER: <image>\n{row['question']}\nASSISTANT:"
        answer = str(row['answer'])

        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        labels = self.processor.tokenizer(
            answer,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        ).input_ids

        return {
            "input_ids": inputs.input_ids.squeeze(),
            "pixel_values": inputs.pixel_values.squeeze(),
            "labels": labels.squeeze()
        }

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    return {
        "input_ids": input_ids,
        "labels": labels,
        "pixel_values": pixel_values
    }

train_dataset = LlavaVQADataset(train_df, processor, max_length=1024)
val_dataset = LlavaVQADataset(val_df, processor, max_length=1024)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=sampler,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    LlavaVQADataset(val_df, processor, max_length=1024),
    batch_size=4,
    collate_fn=collate_fn
)

import bitsandbytes as bnb

# --- 1. MEMORY FIXES ---
model.gradient_checkpointing_enable()
model.config.use_cache = False

# --- 1. OPTIMIZER & SCHEDULER SETUP ---
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-5)

num_epochs = 10
patience = 2
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

# --- 2. FAST TRAINING LOOP ---
history = []
best_val_loss = float('inf')
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True) # Initialize ROUGE scorer

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    model.config.use_cache = False
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

    for batch in train_pbar:
        batch = {k: v.to(model.device, non_blocking=True) for k, v in batch.items()}

        if "pixel_values" in batch and batch["pixel_values"].dtype != torch.bfloat16:
             batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model(**batch).loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        total_train_loss += loss.item()
        train_pbar.set_postfix({"loss": f"{loss.item():.3f}"})

    # --- 3. VALIDATION ---
    model.eval()
    total_val_loss = 0
    epoch_bleu = []
    epoch_rouge = [] # Initialize list for ROUGE scores
    model.config.use_cache = True

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]")
        for i, batch in enumerate(val_pbar):
            batch = {k: v.to(model.device, non_blocking=True) for k, v in batch.items()}

            if "pixel_values" in batch and batch["pixel_values"].dtype != torch.bfloat16:
                batch["pixel_values"] = batch["pixel_values"].to(torch.bfloat16)

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = model(**batch)
            total_val_loss += outputs.loss.item()

            if i < 5:
                generated_ids = model.generate(
                    input_ids=batch['input_ids'],
                    pixel_values=batch['pixel_values'],
                    max_new_tokens=20,
                    use_cache=True
                )
                preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
                for j, pred in enumerate(preds):
                    clean_pred = pred.split("ASSISTANT:")[-1].strip()
                    gt = processor.tokenizer.decode(batch['labels'][j], skip_special_tokens=True)

                    # Calculate BLEU
                    epoch_bleu.append(sentence_bleu([gt.split()], clean_pred.split(), smoothing_function=SmoothingFunction().method1))

                    # Calculate ROUGE
                    rouge_score = scorer.score(gt, clean_pred)['rougeL'].fmeasure
                    epoch_rouge.append(rouge_score)

    model.config.use_cache = False

    metrics = {
        'epoch': epoch + 1,
        'loss': total_train_loss / len(train_loader),
        'eval_loss': total_val_loss / len(val_loader),
        'eval_bleu': np.mean(epoch_bleu) if epoch_bleu else 0,
        'eval_rouge': np.mean(epoch_rouge) if epoch_rouge else 0 # Log ROUGE score
    }
    history.append(metrics)

    if metrics['eval_loss'] < best_val_loss:
        best_val_loss = metrics['eval_loss']
        patience_counter = 0
        model.save_pretrained("./best_fast_model")
        print(f"⭐ New Best Loss: {best_val_loss:.4f} - Model Saved")
    else:
        patience_counter += 1
        print(f"🟡 No improvement. Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("🛑 Early Stopping triggered. Training halted.")
            break


In [ ]:
# @title
import os
import json
import torch
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from transformers import (
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import bitsandbytes as bnb

# --- 1. SETTINGS ---
model_id = "llava-hf/llava-v1.6-mistral-7b-hf"
JSON_PATH = "./archive/VQA_RAD Dataset Public.json"
IMG_DIR = "./archive/VQA_RAD Image Folder/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import torch

# Clear the cache
torch.cuda.empty_cache()

# --- 2. DATASET & COLLATOR ---
class LlavaMedicalCoTDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor
        self.cot_instruction = "Analyze the image step-by-step, then provide the answer."

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(IMG_DIR, row['image_name'])).convert('RGB')
        prompt = f"USER: <image>\nQuestion: {row['question']}\n{self.cot_instruction}\nASSISTANT:"
        inputs = self.processor(text=prompt, images=image, return_tensors="pt")

        # Tokenize Answer as Ground Truth Labels
        labels = self.processor.tokenizer(str(row['answer']), return_tensors="pt", padding=False).input_ids
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "image_sizes": inputs["image_sizes"].squeeze(0),
            "labels": labels.squeeze(0)
        }

def collate_fn(batch):
    input_ids = torch.nn.utils.rnn.pad_sequence([item['input_ids'] for item in batch], batch_first=True, padding_value=processor.tokenizer.pad_token_id)
    labels = torch.nn.utils.rnn.pad_sequence([item['labels'] for item in batch], batch_first=True, padding_value=-100)
    attention_mask = input_ids.ne(processor.tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pixel_values": torch.stack([item['pixel_values'] for item in batch]),
        "image_sizes": torch.stack([item['image_sizes'] for item in batch]),
        "labels": labels
    }

# --- 3. INITIALIZE MODEL ---
processor = LlavaNextProcessor.from_pretrained(model_id)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

model = LlavaNextForConditionalGeneration.from_pretrained(model_id, quantization_config=bnb_config, torch_dtype=torch.bfloat16, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj", "mm_projector"], task_type="CAUSAL_LM"))
model.gradient_checkpointing_enable()

# --- 4. PREPARE DATA LOADERS ---
df = pd.DataFrame(json.load(open(JSON_PATH)))
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_loader = DataLoader(LlavaMedicalCoTDataset(train_df, processor), batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(LlavaMedicalCoTDataset(val_df, processor), batch_size=2, collate_fn=collate_fn)

# --- 5. OPTIMIZER & SCHEDULER ---
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-5)
num_epochs = 10
total_steps = len(train_loader) * num_epochs
# Learning Rate Scheduler with Warmup
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

# --- 6. TRAINING LOOP WITH EARLY STOPPING & HISTORY ---
history = []
best_val_loss = float('inf')
patience = 3
patience_counter = 0

for epoch in range(num_epochs):
    model.train()
    model.config.use_cache = False
    total_train_loss = 0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

    for batch in train_pbar:
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = model(**batch).loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step() # Step the scheduler every batch
        optimizer.zero_grad(set_to_none=True)

        total_train_loss += loss.item()
        train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    # --- VALIDATION ---
    model.eval()
    model.config.use_cache = True
    total_val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                total_val_loss += model(**batch).loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)

    # Save History for plotting
    history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'lr': optimizer.param_groups[0]['lr']
    })

    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")

    # --- EARLY STOPPING LOGIC ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        model.save_pretrained("./best_medical_model")
        print("⭐ New Best Model Saved!")
    else:
        patience_counter += 1
        print(f"🟡 No improvement. Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print("🛑 Early Stopping triggered!")
            break

In [ ]:
# @title
# Save the model and processor
save_path = "./fine_tuned_llava_medical"
model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f"✅ Model and Processor saved to {save_path}")

In [ ]:
# @title
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import textwrap
from PIL import Image

def plot_training_dashboard(history_df):
    """Plots Loss Convergence and Generative Metrics (BLEU/ROUGE)."""
    sns.set_theme(style="whitegrid", palette="muted")
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Panel 1: Loss
    axes[0].plot(history_df['epoch'], history_df['loss'], label='Train Loss', marker='o', lw=2)
    axes[0].plot(history_df['epoch'], history_df['eval_loss'], label='Val Loss', marker='s', ls='--', lw=2)
    axes[0].set_title("Training & Validation Loss", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Cross-Entropy Loss")
    axes[0].legend()

    # Panel 2: NLP Metrics
    if 'eval_bleu' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['eval_bleu'], label='BLEU', marker='^', color='green', lw=2)
        axes[1].plot(history_df['epoch'], history_df['eval_rouge'], label='ROUGE-L', marker='v', color='red', lw=2)
        axes[1].set_title("Generative Quality (BLEU vs ROUGE)", fontsize=14, fontweight='bold')
        axes[1].set_xlabel("Epochs")
        axes[1].set_ylabel("Score (0.0 - 1.0)")
        axes[1].set_ylim(0, 1.0)
        axes[1].legend()

    plt.tight_layout()
    plt.show()



def plot_performance_analysis(results_df):
    """Plots Answer Length Distribution and Category-wise Accuracy."""
    sns.set_theme(style="white")
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Panel 1: Length Distribution (Checks for Hallucinations/Brevity)
    results_df['gt_len'] = results_df['ground_truth'].str.split().str.len()
    results_df['pred_len'] = results_df['prediction'].str.split().str.len()

    sns.kdeplot(results_df['gt_len'], label='Ground Truth', fill=True, ax=axes[0])
    sns.kdeplot(results_df['pred_len'], label='Model Prediction', fill=True, ax=axes[0])
    axes[0].set_title("Answer Length Distribution", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Word Count")
    axes[0].legend()

    # Panel 2: Category Performance (Requires 'phrase_type' column in dataset)
    if 'phrase_type' in results_df.columns:
        cat_perf = results_df.groupby('phrase_type')['bleu'].mean().sort_values()
        sns.barplot(x=cat_perf.values, y=cat_perf.index, palette="viridis", ax=axes[1])
        axes[1].set_title("Average BLEU by Medical Category", fontsize=14, fontweight='bold')
        axes[1].set_xlabel("BLEU Score")

    plt.tight_layout()
    plt.show()



def visualize_predictions_grid(model, processor, test_df, img_dir, num_samples=6):
    """Generates a visual grid of images and predicted text."""
    model.eval()
    samples = test_df.sample(num_samples)
    rows = (num_samples + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(16, 6 * rows))
    axes = axes.flatten()

    for i, (_, row) in enumerate(samples.iterrows()):
        image = Image.open(os.path.join(img_dir, row['image_name'])).convert("RGB")
        prompt = f"USER: <image>\n{row['question']}\nASSISTANT:"
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
            pred = processor.decode(out[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

        axes[i].imshow(image)
        axes[i].axis('off')

        # Color coding: Green if correct keyword exists in prediction
        is_correct = any(word in pred.lower() for word in str(row['answer']).lower().split())
        title_color = '#2ca02c' if is_correct else '#d62728'

        title = f"Q: {textwrap.fill(row['question'], 50)}\nPred: {textwrap.fill(pred, 50)}\nGT: {row['answer']}"
        axes[i].set_title(title, fontsize=10, color=title_color, loc='left', fontweight='bold')

    for j in range(i + 1, len(axes)): axes[j].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# @title
# --- STEP 1: Generate Training History Plot ---
history_df = pd.DataFrame(history)
plot_training_dashboard(history_df)

# --- STEP 2: Generate Evaluation Results DataFrame ---
print("Generating evaluation results for analysis...")
eval_list = []
model.eval()
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# Evaluate on a sample of the test set for plotting
for _, row in tqdm(test_df.sample(50).iterrows(), total=50):
    image = Image.open(os.path.join(IMG_DIR, row['image_name'])).convert("RGB")
    prompt = f"USER: <image>\n{row['question']}\nASSISTANT:"
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=32)
        pred = processor.decode(out[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

    gt = str(row['answer'])
    bleu = sentence_bleu([gt.split()], pred.split(), smoothing_function=SmoothingFunction().method1)
    rouge = scorer.score(gt, pred)['rougeL'].fmeasure

    # Store everything including metadata for categorical plotting
    eval_list.append({
        'ground_truth': gt,
        'prediction': pred,
        'bleu': bleu,
        'rouge': rouge,
        'phrase_type': row.get('phrase_type', 'General') # Metadata
    })

# During validation, group your results
results_df = pd.DataFrame(eval_list)

# Check BLEU score per category
category_stats = results_df.groupby('phrase_type')['bleu'].mean()
print("--- Performance by Category ---")
print(category_stats)

# --- STEP 3: Trigger Final Plots ---
plot_performance_analysis(results_df)
visualize_predictions_grid(model, processor, test_df, IMG_DIR, num_samples=6)

In [ ]:
# @title
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
import os
from PIL import Image
from tqdm.auto import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def plot_final_results(history_list):
    """Plots training/val loss and generative metrics from your history."""
    df = pd.DataFrame(history_list)
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot 1: Loss
    axes[0].plot(df['epoch'], df['loss'], label='Train Loss', marker='o', lw=2)
    axes[0].plot(df['epoch'], df['eval_loss'], label='Val Loss', marker='s', ls='--', lw=2)
    axes[0].set_title("Training & Validation Loss", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Epochs")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    # Plot 2: Generative Metrics (BLEU/ROUGE)
    if 'eval_bleu' in df.columns:
        axes[1].plot(df['epoch'], df['eval_bleu'], label='BLEU (Similarity)', color='green', marker='^', lw=2)
        axes[1].plot(df['epoch'], df['eval_rouge'], label='ROUGE-L (Recall)', color='red', marker='v', lw=2)
        axes[1].set_title("Generative Performance Metrics", fontsize=14, fontweight='bold')
        axes[1].set_xlabel("Epochs")
        axes[1].set_ylabel("Score (0.0 - 1.0)")
        axes[1].set_ylim(0, 1.0)
        axes[1].legend()

    plt.tight_layout()
    plt.show()



def run_test_and_visual_grid(model, processor, test_df, img_dir, num_samples=6):
    """Calculates final scores on test set and displays a visual grid."""
    model.eval()

    # --- PART A: Quantitative Metrics ---
    print("Calculating final metrics on test set...")
    bleu_scores = []
    smoothing = SmoothingFunction().method1

    # We sample 50 for a quick final score, or use len(test_df) for full test
    test_sample = test_df.sample(min(50, len(test_df)))

    for _, row in tqdm(test_sample.iterrows(), total=len(test_sample)):
        image = Image.open(os.path.join(img_dir, row['image_name'])).convert("RGB")
        prompt = f"USER: <image>\n{row['question']}\nASSISTANT:"
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
            prediction = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

        gt = str(row['answer'])
        score = sentence_bleu([gt.split()], prediction.split(), smoothing_function=smoothing)
        bleu_scores.append(score)

    print(f"\n🎯 Final Test BLEU Score: {np.mean(bleu_scores):.4f}")

    # --- PART B: Visual Grid ---
    samples = test_df.sample(num_samples)
    rows = (num_samples + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(16, 6 * rows))
    axes = axes.flatten()

    for i, (_, row) in enumerate(samples.iterrows()):
        image = Image.open(os.path.join(img_dir, row['image_name'])).convert("RGB")
        prompt = f"USER: <image>\n{row['question']}\nASSISTANT:"
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=64, temperature=0.1)
            prediction = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

        axes[i].imshow(image)
        axes[i].axis('off')

        # Determine if answer is roughly correct (for color coding)
        is_correct = prediction.lower() in str(row['answer']).lower() or str(row['answer']).lower() in prediction.lower()
        title_color = '#2ca02c' if is_correct else '#d62728'

        title = f"Q: {textwrap.fill(row['question'], 45)}\nPred: {textwrap.fill(prediction, 45)}\nGT: {row['answer']}"
        axes[i].set_title(title, fontsize=10, color=title_color, loc='left', fontweight='bold')

    for j in range(i + 1, len(axes)): axes[j].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# @title
# 1. Plot the training history you captured earlier
if 'history' in globals():
    plot_final_results(history)
else:
    print("No history found to plot.")

# 2. Run the quantitative evaluation and the visual grid
run_test_and_visual_grid(
    model=model,
    processor=processor,
    test_df=test_df,
    img_dir=IMG_DIR,
    num_samples=6
)

In [ ]:
# @title
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2
!pip install evaluate rouge_score nltk

In [ ]:
# @title
import os
import json
import torch
import pandas as pd
from PIL import Image
from datasets import Dataset
from unsloth import FastVisionModel
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
from transformers import TrainerCallback
import evaluate
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback
import numpy as np
import time

# Load metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

class StatusUpdateCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.start_time = time.time()  # Initialize start time

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            # Calculate elapsed time manually
            elapsed = time.time() - self.start_time

            loss = logs.get("loss", "N/A")
            lr = logs.get("learning_rate", "N/A")

            # Format numbers safely in case they are strings
            loss_fmt = f"{loss:.4f}" if isinstance(loss, (int, float)) else loss
            lr_fmt = f"{lr:.2e}" if isinstance(lr, (int, float)) else lr

            print(f"🚩 [Time: {elapsed:.2f}s] Step: {state.global_step} | Loss: {loss_fmt} | LR: {lr_fmt}")

# --- SETTINGS ---
# Using the optimized Qwen3 or Qwen2.5 vision model from Unsloth
model_name = "unsloth/Qwen2.5-VL-7B-Instruct-unsloth-bnb-4bit"
JSON_PATH = "./archive/VQA_RAD Dataset Public.json"
IMG_DIR = "./archive/VQA_RAD Image Folder/"
OUTPUT_DIR = "./vqa_med_results"

# 1. LOAD MODEL & TOKENIZER
# --- 1. MODEL & LORA SETUP ---
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = model_name,
    load_in_4bit = True, # 4-bit quantization for efficiency
)
model = FastVisionModel.get_peft_model(
    model,
    r = 16, # LoRA Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

import pandas as pd
from datasets import Dataset
import numpy as np

def compute_metrics(prepared_eval_preds):
    logits, labels = prepared_eval_preds

    # 1. Convert Logits to Token IDs
    # If logits is 3D (batch, seq, vocab), we take the argmax over the last dim
    if isinstance(logits, np.ndarray) and len(logits.shape) == 3:
        preds = np.argmax(logits, axis=-1)
    else:
        preds = logits # Already IDs

    # 2. Robust Decoding Loop
    decoded_preds = []
    for p in preds:
        # We ensure 'p' is treated as a flat array
        p_flat = np.array(p).flatten()
        valid_p = p_flat[p_flat != -100]
        decoded_preds.append(tokenizer.decode(valid_p, skip_special_tokens=True))

    decoded_labels = []
    for l in labels:
        l_flat = np.array(l).flatten()
        valid_l = l_flat[l_flat != -100]
        decoded_labels.append(tokenizer.decode(valid_l, skip_special_tokens=True))

    # 3. Requirement: Check for 'Reasoning' (CoT) presence
    # Let's track if the model actually generated the CoT section
    cot_hits = sum(1 for text in decoded_preds if "reasoning" in text.lower())

    return {
        "avg_gen_len": np.mean([len(p) for p in decoded_preds]),
        "cot_compliance_rate": cot_hits / len(decoded_preds) if decoded_preds else 0
    }

# 1. Load the raw JSON
with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

# Convert to HF Dataset format
def prepare_vqa_rad(example):
    img_path = os.path.join(IMG_DIR, example['image_name'])
    image = Image.open(img_path).convert("RGB")

    # multi-modal format required by Unsloth vision models
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"Question: {example['question']}\nAnalyze the image step-by-step, then provide the answer."},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": example['answer']},
                ],
            },
        ]
    }

# 2. Convert to Pandas first (Pandas handles mixed types better than Arrow)
df = pd.DataFrame(raw_data)

# 3. SANITIZE TYPES: Force problematic columns to string
# This fixes the "Expected bytes, got int" error
for col in df.columns:
    df[col] = df[col].astype(str)

# 4. Convert the cleaned DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# 5. Now apply the mapping (this will handle the images)
dataset = dataset.map(prepare_vqa_rad, remove_columns=dataset.column_names)

# Split for training
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset_split["train"]
val_data = dataset_split["test"]

In [ ]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback
from PIL import Image
import io
import gc
gc.collect()

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = []

    for convo in convos:
        # 1. Standardize to a list of messages
        messages = [convo] if isinstance(convo, dict) else convo

        for message in messages:
            if message.get("role") == "user":
                content = message.get("content", [])

                # Case A: Content is a simple string
                if isinstance(content, str):
                    if "think step by step" not in content.lower():
                        message["content"] += "\nRespond with a 'Reasoning' section followed by a 'Final Answer'. Let's think step by step."

                # Case B: Content is a list (Multi-modal)
                elif isinstance(content, list):
                    for item in content:
                        if not isinstance(item, dict): continue

                        # Requirement: Inject Medical CoT
                        text_val = item.get("text")
                        if isinstance(text_val, str):
                            if "think step by step" not in text_val.lower():
                                item["text"] += "\nRespond with a 'Reasoning' section followed by a 'Final Answer'. Let's think step by step."

                        # Requirement: Process Image Bytes
                        if "image" in item and item["image"] is not None:
                            if isinstance(item["image"], dict) and "bytes" in item["image"]:
                                img_bytes = item["image"]["bytes"]
                                item["image"] = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        # 2. Apply the Chat Template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    # FIX: Return the list of strings directly, NOT a dictionary
    return texts
# 3. CONFIGURE TRAINING
training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    num_train_epochs = 10,
    learning_rate = 2e-4, # Initial Learning Rate
    lr_scheduler_type = "cosine", # Learning Rate Scheduler
    warmup_ratio = 0.1,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    eval_strategy = "steps", # Evaluate during training
    eval_steps = 10,
    save_strategy = "steps",
    save_steps = 10,
    load_best_model_at_end = True, # Required for Early Stopping
    metric_for_best_model = "loss",
    save_total_limit = 2,
    report_to = "none",
    # predict_with_generate = True, # CRITICAL: Enables generation during eval
    # generation_max_length = 64, # Limit length to save time
    optim = "adamw_8bit",
    remove_unused_columns = True,
    weight_decay = 0.01,
    dataloader_pin_memory = True, # Works with non_blocking
    gradient_checkpointing = True,
)
model.config.use_cache = False

# Initialize Trainer with Early Stopping (Patience = 3)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = val_data,
    max_seq_length = 4096,
    args = training_args,
    compute_metrics = compute_metrics, # Pass the function here
    # Add the new callback here alongside EarlyStopping
    callbacks = [
        EarlyStoppingCallback(early_stopping_patience=3),
        StatusUpdateCallback()
    ],
    formatting_func = formatting_prompts_func,
)

# 4. START TRAINING
trainer_stats = trainer.train()

# 5. SAVE HISTORY FOR PLOTTING
history = trainer.state.log_history
with open("training_history.json", "w") as f:
    json.dump(history, f)

print("✅ Training complete. History saved to training_history.json")

In [ ]:
# @title
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

with open("training_history.json", "r") as f:
    history = json.load(f)

df = pd.DataFrame(history)
eval_df = df[df['eval_loss'].notna()].copy()

sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- PANEL 1: Loss ---
sns.lineplot(data=df[df['loss'].notna()], x='step', y='loss', ax=ax1, label='Train Loss')
sns.scatterplot(data=eval_df, x='step', y='eval_loss', ax=ax1, color='red', label='Val Loss')
ax1.set_title("Training vs Validation Loss")

# --- PANEL 2: BLEU & ROUGE ---
if 'eval_bleu' in eval_df.columns:
    ax2_twin = ax2.twinx()
    sns.lineplot(data=eval_df, x='step', y='eval_bleu', ax=ax2, color='blue', marker='o', label='BLEU')
    sns.lineplot(data=eval_df, x='step', y='eval_rougeL', ax=ax2_twin, color='green', marker='s', label='ROUGE-L')

    ax2.set_ylabel("BLEU Score", color='blue')
    ax2_twin.set_ylabel("ROUGE-L Score", color='green')
    ax2.set_title("NLP Evaluation Metrics (Validation)")
    ax2.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# @title
model.save_pretrained("vqa_med_qwen_lora")
tokenizer.save_pretrained("vqa_med_qwen_lora")
from unsloth import FastVisionModel
import torch
from PIL import Image

# 1. Load for Inference (If starting a fresh session)
# model, tokenizer = FastVisionModel.from_pretrained("vqa_med_qwen_lora", load_in_4bit=True)

FastVisionModel.for_inference(model) # Enable 2x faster inference

def ask_medical_question(image_path, question):
    image = Image.open(image_path).convert("RGB")

    # Format according to Qwen-VL template
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": f"Question: {question}\nAnalyze the image step-by-step, then provide the answer."},
            ],
        }
    ]

    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=128,
        use_cache=True,
        temperature=0.1, # Low temperature for medical accuracy
        top_p=0.9
    )

    prediction = tokenizer.decode(output[0], skip_special_tokens=True)
    # Extract only the Assistant's response
    return prediction.split("assistant")[-1].strip()

# --- RUN TEST ---
sample_row = val_data[0] # Test on the first item of your validation set
img_name = sample_row['messages'][0]['content'][0]['image'] # Access original image
q = "What abnormality is visible in this scan?"

print(f"Question: {q}")
print(f"Model Answer: {ask_medical_question(img_name, q)}")

In [ ]:
# @title
# 1. Switch to Inference Mode
FastVisionModel.for_inference(model)
model.config.use_cache = True # Re-enable cache for fast generation

def inference_session(image_path, question):
    image = Image.open(image_path).convert("RGB")

    # Chain of Thought Prompting
    prompt = f"Question: {question}\nInstruction: Let's think step by step. Analyze the anatomical features first."

    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(image, input_text, return_tensors="pt").to("cuda", non_blocking=True)

    # Inference with your specific params
    output = model.generate(
        **inputs,
        max_new_tokens = 128,
        temperature = 0,             # Greedy decoding for facts
        repetition_penalty = 1.2,    # Prevent looping
        use_cache = True,
        do_sample = False            # Required when temp=0
    )

    return tokenizer.decode(output[0], skip_special_tokens=True).split("assistant")[-1].strip()
def generate_medical_response(image, prompt):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Process image and text into tensors
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        # REQUIREMENT: Temperature 0 (Deterministic/Greedy)
        temperature=0.0,
        do_sample=False,
        # REQUIREMENT: Repetition Penalty (Prevents looping)
        repetition_penalty=1.1,
        use_cache=True
    )
    return processor.batch_decode(output_ids, skip_special_tokens=True)
# --- Example Test ---
test_img = "./archive/VQA_RAD Image Folder/synpic51454.jpg"
print(inference_session(test_img, "What is the primary finding in this chest X-ray?"))